In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from glob import glob

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import albumentations as A

tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
from glob import glob

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import albumentations as A

tf.random.set_seed(42)
np.random.seed(42)

### Data Augmentation

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 16

def extract_box(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    # Get image dimensions
    size = root.find('size')
    width = float(size.find('width').text)
    height = float(size.find('height').text)
    
    # Get bounding box
    bndbox = root.find('.//bndbox')
    xmin = float(bndbox.find('xmin').text)
    ymin = float(bndbox.find('ymin').text)
    xmax = float(bndbox.find('xmax').text)
    ymax = float(bndbox.find('ymax').text)
    
    return [xmin, ymin, xmax, ymax], width, height

def get_data(data_dir):
    images = []
    boxes = []
    
    xml_files = glob(os.path.join(data_dir, '*.xml'))
    for xml_file in xml_files:
        img_file = xml_file.replace('.xml', '.jpg')
        if not os.path.exists(img_file):
            img_file = xml_file.replace('.xml', '.png') # Fallback to png
            
        if os.path.exists(img_file):
            # Load and convert image to RGB
            img = cv2.imread(img_file)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Extract coordinates
            box, w, h = extract_box(xml_file)
            
            images.append(img)
            
            # We keep the raw [xmin, ymin, xmax, ymax] for albumentations
            boxes.append([box])
            
    return images, boxes

TRAIN_DIR = 'dataset/ARLP Datasets ver-2/train/'
TEST_DIR = 'dataset/ARLP Datasets ver-2/test/'

train_images, train_boxes = get_data(TRAIN_DIR)
val_images, val_boxes = get_data(TEST_DIR)

print(f"Loaded {len(train_images)} training images and {len(val_images)} validation images.")

### Albumentations generator

In [ ]:
class BoundingBoxDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, images, boxes, batch_size, augment=False):
        self.images = images
        self.boxes = boxes
        self.batch_size = batch_size
        
        # Albumentations pipeline
        if augment:
            self.transform = A.Compose([
                A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10, p=0.5, border_mode=cv2.BORDER_CONSTANT)
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
        else:
            # Only resize for validation
            self.transform = A.Compose([
                A.Resize(IMAGE_SIZE, IMAGE_SIZE)
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    def __len__(self):
        return int(np.ceil(len(self.images) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_images = self.images[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_boxes = self.boxes[idx * self.batch_size:(idx + 1) * self.batch_size]
        
        X = np.zeros((len(batch_images), IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.float32)
        y = np.zeros((len(batch_images), 4), dtype=np.float32)
        
        for i in range(len(batch_images)):
            transformed = self.transform(
                image=batch_images[i],
                bboxes=batch_boxes[i],
                labels=[1] # Dummy label
            )
            
            # Normalize image to [0, 1]
            X[i] = transformed['image'] / 255.0
            
            # If box disappeared after cropping/augmenting, use a default centered box to prevent NaN
            if len(transformed['bboxes']) == 0:
                box = [0, 0, IMAGE_SIZE, IMAGE_SIZE]
            else:
                box = transformed['bboxes'][0]
                
            # Normalize bounding box to [0, 1] relative to the RESIZED image
            y[i] = [box[0]/IMAGE_SIZE, box[1]/IMAGE_SIZE, box[2]/IMAGE_SIZE, box[3]/IMAGE_SIZE]
            
        return X, y

train_gen = BoundingBoxDataGenerator(train_images, train_boxes, BATCH_SIZE, augment=True)
val_gen = BoundingBoxDataGenerator(val_images, val_boxes, BATCH_SIZE, augment=False)

### MobileNetV2 Architecture

In [ ]:
def build_model():
    # Load pretrained MobileNetV2
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))
    
    # Freeze the backbone (it acts as a feature extractor)
    base_model.trainable = False
    
    # Add our custom regression head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)
    
    # 4 output coordinates (xmin, ymin, xmax, ymax) normalized between 0 and 1
    predictions = Dense(4, activation='sigmoid', name='bounding_box')(x)
    
    model = Model(inputs=base_model.input, outputs=predictions, name="ARLP_MobileNetV2")
    return model

model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='mse')
model.summary()

### training

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint('ARLP-MobileNetV2.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

# history = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=100,
#     callbacks=callbacks
# )